# Export Model Scores to ROOT

Consolidates every model's out-of-fold (OOF) score CSV into **new** ROOT
files - one per (channel, run) combination - with one branch per model:
`score_xgboost`, `score_mlp`, `score_dnn`, `score_gnn`, `score_pnn`. This
does **not** touch the original per-process input ROOT files
(`signal_ggF.root`, `ttbar.root`, ...) - it only reads each model's own
`oof_scores_*.csv` artifact (already keyed by `eventNumber`) and writes a
brand-new file `scores_all_models_<run>.root` alongside them.

**Scope**: `channel` in `{1l2tau, 2l2tau}` x `run` in `{run2, run3}` (4
combinations). `Combined`-track OOF is not included here - if a per-run
score isn't available for a model, that column is simply omitted for that
(channel, run) (never silently fabricated).

**Per-model source files, by convention** (see repo memory
`mva_project_facts.md` for the collision-bug history behind these
suffixes):
- XGBoost (master pipeline): `oof_scores_<run>.csv` (unsuffixed - the
  canonical/original artifact every other model's suffix was chosen to
  avoid colliding with).
- MLP: `oof_scores_<run>_mlp.csv`
- DNN: `oof_scores_<run>_dnn.csv`
- GNN: `oof_scores_<run>_gnn.csv`
- PNN: a single **channel-combined** file,
  `PPSSP_2026/pnn_channel_combined/combined/oof_scores_combined.csv`,
  covering both channels and both runs at once (has its own `channel`/`run`
  columns) - sliced per (channel, run) here rather than loaded per-track.

**Merge key**: `eventNumber` (outer join across whichever models have a
file for that (channel, run) - an event missing from one model's OOF
because it doesn't exist gets `NaN` in that model's score column, not
dropped). `process`/`label`/`w_phys`/`fold` are carried over once from
whichever source has them (preferring XGBoost's, since it is present for
the most combinations), not duplicated per model.

**Caveat as of the time this was written**: most model notebooks
(MLP/DNN/GNN, all tracks; XGBoost 1l2tau Run 3 + Combined) have NOT been
executed yet with the current k-fold+Optuna pipeline, so several
`score_<model>` columns will be entirely missing/empty until those
notebooks are actually run - this script reports exactly which
model/(channel,run) combinations were found vs missing, it does not hide
gaps.


In [1]:
import warnings
from pathlib import Path

import pandas as pd
import uproot

BASE = Path("PPSSP_2026")
CHANNELS = ["1l2tau", "2l2tau"]
RUNS = ["run2", "run3"]

# Per-model OOF filename suffix convention (see markdown above / repo memory
# for the collision-bug history that produced these). XGBoost is the
# canonical/unsuffixed one; every other model was suffixed specifically to
# avoid overwriting it.
MODEL_SUFFIXES = {
    "xgboost": "",
    "mlp": "_mlp",
    "dnn": "_dnn",
    "gnn": "_gnn",
}

PNN_COMBINED_CSV = BASE / "pnn_channel_combined" / "combined" / "oof_scores_combined.csv"
CONTEXT_COLS = ["process", "label", "fold", "w_phys"]  # carried over once, not per-model

In [4]:
_pnn_cache = None


def load_pnn_scores(channel, run):

    """
    PNN.ipynb writes ONE channel-combined OOF file covering both channels
    and both runs at once (columns include `channel`/`run`), rather than a
    separate file per (channel, run) like every other model - so instead of
    a filename lookup, slice the single cached CSV by channel/run. Returns
    None if the file doesn't exist yet, or an empty-filtered slice if the
    file exists but has no rows for this (channel, run).
    """

    global _pnn_cache

    if not PNN_COMBINED_CSV.exists():
        return None

    if _pnn_cache is None:
        _pnn_cache = pd.read_csv(PNN_COMBINED_CSV)

    run_num = int(run.replace("run", ""))
    sub = _pnn_cache.loc[
        (_pnn_cache["channel"] == channel) & (_pnn_cache["run"] == run_num)
    ]
    return sub[["eventNumber", *CONTEXT_COLS, "score"]].copy() if len(sub) else None


def load_model_scores(channel, run):

    """
    For one (channel, run) combination: load whichever per-model OOF CSVs
    exist (XGBoost/MLP/DNN/GNN via filename suffix, PNN via the shared
    combined CSV sliced above), rename each `score` column to
    `score_<model>`, and outer-merge them all on `eventNumber`. Context
    columns (process/label/fold/w_phys) are carried over ONCE, taken from
    whichever source has them first (XGBoost preferred, since it's present
    for the most combinations currently).

    Returns (merged_df, found_models, missing_models).
    """

    base_dir = BASE / channel / run
    found, missing = [], []
    merged = None

    for model, suffix in MODEL_SUFFIXES.items():
        csv_path = base_dir / f"oof_scores_{run}{suffix}.csv"
        if not csv_path.exists():
            missing.append(model)
            continue

        df = pd.read_csv(csv_path)
        keep_cols = ["eventNumber"] + [c for c in CONTEXT_COLS if c in df.columns] + ["score"]
        df = df[keep_cols].rename(columns={"score": f"score_{model}"})
        found.append(model)

        if merged is None:
            merged = df
        else:
            context_overlap = [c for c in CONTEXT_COLS if c in df.columns and c in merged.columns]
            merged = merged.merge(
                df.drop(columns=context_overlap), on="eventNumber", how="outer",
            )

    pnn_df = load_pnn_scores(channel, run)
    if pnn_df is not None:
        pnn_df = pnn_df.rename(columns={"score": "score_pnn"})
        found.append("pnn")
        if merged is None:
            merged = pnn_df
        else:
            context_overlap = [c for c in CONTEXT_COLS if c in pnn_df.columns and c in merged.columns]
            merged = merged.merge(
                pnn_df.drop(columns=context_overlap), on="eventNumber", how="outer",
            )
    else:
        missing.append("pnn")

    return merged, found, missing


def write_scores_root(df, out_path, tree_name="scores"):

    """Write `df` to a NEW ROOT file (uproot.recreate - never touches an
    existing file's original content, just creates/overwrites this one
    export file) as a flat TTree, one branch per column. String columns
    (e.g. `process`) are converted via numpy's OWN .astype(str) (fixed-
    width unicode dtype) rather than pandas' astype(str).to_numpy() (which
    stays object-dtype) - uproot/awkward can't write object-dtype arrays."""

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    branches = {}
    for col in df.columns:
        vals = df[col].to_numpy()
        if vals.dtype == object:
            vals = vals.astype(str)
        branches[col] = vals

    with uproot.recreate(out_path) as f:
        f[tree_name] = branches

    print(f"Wrote {len(df)} rows, {len(df.columns)} branches -> {out_path}")

In [5]:
summary_rows = []

for channel in CHANNELS:
    for run in RUNS:
        merged, found, missing = load_model_scores(channel, run)

        if merged is None:
            print(f"[{channel}/{run}] NO model scores found at all (found=[], missing={missing}) - skipped, no file written.")
            summary_rows.append({"channel": channel, "run": run, "n_events": 0,
                                  "models_found": ", ".join(found) or "-", "models_missing": ", ".join(missing)})
            continue

        out_path = BASE / channel / run / f"scores_all_models_{run}.root"
        write_scores_root(merged, out_path)

        print(f"[{channel}/{run}] models found: {found} | models missing: {missing}")
        if missing:
            warnings.warn(f"[{channel}/{run}] missing OOF scores for: {missing} - "
                           f"those score_<model> columns are absent from this export "
                           f"until the corresponding notebook is executed.")

        summary_rows.append({"channel": channel, "run": run, "n_events": len(merged),
                              "models_found": ", ".join(found) or "-", "models_missing": ", ".join(missing) or "-"})

summary = pd.DataFrame(summary_rows)
print("\nExport summary:")
summary

Wrote 701664 rows, 7 branches -> PPSSP_2026/1l2tau/run2/scores_all_models_run2.root
[1l2tau/run2] models found: ['xgboost', 'pnn'] | models missing: ['mlp', 'dnn', 'gnn']


/tmp/ipykernel_325901/3288770573.py:18: UserWarning: [1l2tau/run2] missing OOF scores for: ['mlp', 'dnn', 'gnn'] - those score_<model> columns are absent from this export until the corresponding notebook is executed.
  warnings.warn(f"[{channel}/{run}] missing OOF scores for: {missing} - "


Wrote 1324892 rows, 6 branches -> PPSSP_2026/1l2tau/run3/scores_all_models_run3.root
[1l2tau/run3] models found: ['pnn'] | models missing: ['xgboost', 'mlp', 'dnn', 'gnn']
Wrote 95370 rows, 7 branches -> PPSSP_2026/2l2tau/run2/scores_all_models_run2.root
[2l2tau/run2] models found: ['xgboost', 'pnn'] | models missing: ['mlp', 'dnn', 'gnn']


/tmp/ipykernel_325901/3288770573.py:18: UserWarning: [1l2tau/run3] missing OOF scores for: ['xgboost', 'mlp', 'dnn', 'gnn'] - those score_<model> columns are absent from this export until the corresponding notebook is executed.
  warnings.warn(f"[{channel}/{run}] missing OOF scores for: {missing} - "
/tmp/ipykernel_325901/3288770573.py:18: UserWarning: [2l2tau/run2] missing OOF scores for: ['mlp', 'dnn', 'gnn'] - those score_<model> columns are absent from this export until the corresponding notebook is executed.
  warnings.warn(f"[{channel}/{run}] missing OOF scores for: {missing} - "


Wrote 231118 rows, 7 branches -> PPSSP_2026/2l2tau/run3/scores_all_models_run3.root
[2l2tau/run3] models found: ['xgboost', 'pnn'] | models missing: ['mlp', 'dnn', 'gnn']

Export summary:


/tmp/ipykernel_325901/3288770573.py:18: UserWarning: [2l2tau/run3] missing OOF scores for: ['mlp', 'dnn', 'gnn'] - those score_<model> columns are absent from this export until the corresponding notebook is executed.
  warnings.warn(f"[{channel}/{run}] missing OOF scores for: {missing} - "


,channel,run,n_events,models_found,models_missing
0,1l2tau,run2,701664,"xgboost, pnn","mlp, dnn, gnn"
1,1l2tau,run3,1324892,pnn,"xgboost, mlp, dnn, gnn"
2,2l2tau,run2,95370,"xgboost, pnn","mlp, dnn, gnn"
3,2l2tau,run3,231118,"xgboost, pnn","mlp, dnn, gnn"
